# World War II Aerial Bombing Operations (THOR Database)
### Comprehensive Cross-Theater Data Exploration & Geospatial Analysis

This notebook performs an exhaustive investigation of the complete **178,281 attack records** recorded during World War II in the United States Department of Defense / Air Force Research Institute (AFRI) **Theater History of Operations (THOR)** database.

**Key Analyses Included:**
1. **Theaters of War**: European Theater (ETO), Mediterranean (MTO), Pacific (PTO), China-Burma-India (CBI).
2. **Tunisia & North Africa Campaign (1942-1943)**: Strategic interdiction, tactical strikes, ports (Tunis, Bizerta, Sfax, Sousse, Gabes).
3. **Strategic Bombing in Europe**: Industrial hubs, marshalling yards, refineries in Germany and occupied Europe.
4. **Pacific Theater & Island Hopping**: Philippines, Marianas, B-29 incendiary and atomic raids against Japan.
5. **Air Fleet & Ordnance Breakdown**: B-17 vs B-24 vs Lancaster vs B-29, HE vs Incendiary firebombing.

In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap

# Connect to the pre-indexed SQLite database
conn = sqlite3.connect('../data/processed/thor_wwii.sqlite')
print('Connected to thor_wwii.sqlite successfully!')

## 1. Global Bomb Tonnage by Theater of Operations

In [2]:
query = '''
SELECT 
    THEATER, 
    COUNT(*) as missions,
    ROUND(SUM(total_tons_clean), 1) as total_tons,
    ROUND(SUM(CASE WHEN tonnage_outlier_reason IS NULL THEN TONS_OF_HE END), 1) as he_tons,
    ROUND(SUM(CASE WHEN tonnage_outlier_reason IS NULL THEN TONS_OF_IC END), 1) as ic_tons,
    ROUND(SUM(CASE WHEN tonnage_outlier_reason IS NULL THEN TONS_OF_FRAG END), 1) as frag_tons
FROM missions
GROUP BY THEATER
ORDER BY total_tons DESC;
'''
theater_df = pd.read_sql_query(query, conn)
theater_df

## 2. Tunisia & North Africa Campaign Deep-Dive (1942–1943)

In [3]:
tunisia_query = '''
SELECT 
    TGT_LOCATION as location,
    COUNT(*) as missions,
    ROUND(SUM(total_tons_clean), 1) as total_tons,
    MIN(mission_date_iso) as first_strike,
    MAX(mission_date_iso) as last_strike,
    ROUND(AVG(target_lat), 4) as lat,
    ROUND(AVG(target_lon), 4) as lon
FROM missions
WHERE UPPER(TGT_COUNTRY) = 'TUNISIA' AND has_valid_target_coords = 1
GROUP BY TGT_LOCATION
ORDER BY total_tons DESC
LIMIT 15;
'''
tunisia_df = pd.read_sql_query(tunisia_query, conn)
tunisia_df

In [4]:
# Interactive Map of Bombing Targets in Tunisia
m_tunisia = folium.Map(location=[35.5, 10.0], zoom_start=7, tiles='CartoDB dark_matter')
for _, row in tunisia_df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=max(5, min(25, (row['total_tons'] ** 0.5) * 0.4)),
        popup=f"<b>{row['location']}</b><br>Missions: {row['missions']:,}<br>Total Tons: {row['total_tons']:,.1f} t",
        color='#f97316',
        fill=True,
        fill_color='#f97316',
        fill_opacity=0.7
    ).add_to(m_tunisia)
m_tunisia

## 3. Top Bombing Targets in Europe (ETO)

In [5]:
eto_targets = pd.read_sql_query('''
SELECT target_country, target_location, mission_count, total_tons, first_mission, last_mission
FROM targets_summary
WHERE theater = 'ETO'
ORDER BY total_tons DESC
LIMIT 15;
''', conn)
eto_targets

## 4. Pacific Theater & B-29 Superfortress Operations

In [6]:
b29_query = '''
SELECT 
    TGT_LOCATION as target,
    COUNT(*) as missions,
    ROUND(SUM(total_tons_clean), 1) as total_tons,
    ROUND(SUM(CASE WHEN tonnage_outlier_reason IS NULL THEN TONS_OF_IC END), 1) as incendiary_tons,
    MIN(mission_date_iso) as first_raid,
    MAX(mission_date_iso) as last_raid
FROM missions
WHERE aircraft_full_name LIKE '%B-29%'
GROUP BY TGT_LOCATION
ORDER BY total_tons DESC
LIMIT 12;
'''
b29_df = pd.read_sql_query(b29_query, conn)
b29_df